# Mới

In [ ]:
!pip install transformers torch onnx onnxruntime pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 78.8 MB/s eta 0:00:00


In [ ]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.3/159.3 kB 11.2 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
from transformers import CLIPModel
from onnxruntime.quantization import quantize_dynamic, QuantType
import os

MODEL_ID = "openai/clip-vit-base-patch32"
ONNX_PATH = "scripts/artifacts/clip_vision_512.onnx"
QUANT_PATH = "scripts/artifacts/clip_vision_quant.onnx"
os.makedirs("scripts/artifacts", exist_ok=True)

# ---------------------------
# 1️⃣ Wrapper chuẩn production
# ---------------------------
class CLIPVisionProduction(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.vision_model = model.vision_model
        self.visual_projection = model.visual_projection

    def forward(self, pixel_values):
        vision_outputs = self.vision_model(pixel_values=pixel_values)

        # 768-dim pooled output
        pooled = vision_outputs.pooler_output

        # Projection 768 → 512
        embeddings = self.visual_projection(pooled)

        # Normalize ngay trong graph (cosine space)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)

        return embeddings


print("⏳ Loading CLIP model...")
model = CLIPModel.from_pretrained(MODEL_ID)
model.eval()

wrapper = CLIPVisionProduction(model)

dummy_input = torch.randn(1, 3, 224, 224)

print("⏳ Exporting ONNX (batch fixed = 1)...")

with torch.no_grad():
    torch.onnx.export(
      wrapper,
      dummy_input,
      ONNX_PATH,
      input_names=["pixel_values"],
      output_names=["image_embeddings"],
      opset_version=18,
      do_constant_folding=True,
      export_params=True,
      keep_initializers_as_inputs=False,
      dynamo=False   # ← CỰC KỲ QUAN TRỌNG
  )


print(f"✅ Exported: {ONNX_PATH}")

# ---------------------------
# 2️⃣ Cleanup RAM
# ---------------------------
print("model.logit_scale", model.logit_scale)
del model
del wrapper
torch.cuda.empty_cache()

# ---------------------------
# 4️⃣ File size check
# ---------------------------
print("\n📦 File sizes:")
print("Original:", round(os.path.getsize(ONNX_PATH)/1024/1024, 2), "MB")


⏳ Loading CLIP model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipython-input-2297245095.py:46: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_on

⏳ Exporting ONNX (batch fixed = 1)...
✅ Exported: clip_vision_512.onnx
model.logit_scale Parameter containing:
tensor(4.6052, requires_grad=True)


TypeError: CLIPModel.get_image_features() missing 1 required positional argument: 'pixel_values'

In [ ]:
import torch
import numpy as np
import onnxruntime as ort
from transformers import CLIPModel, CLIPProcessor
from PIL import Image

MODEL_ID = "openai/clip-vit-base-patch32"
ONNX_PATH = "scripts/artifacts/clip_vision_512.onnx"

# -----------------------------
# 1️⃣ Load HF model
# -----------------------------
print("Loading HF model...")
model = CLIPModel.from_pretrained(MODEL_ID)
model.eval()

processor = CLIPProcessor.from_pretrained(MODEL_ID)

# -----------------------------
# 2️⃣ Dummy image input
# -----------------------------
# Tạo ảnh trắng dummy
image = Image.new("RGB", (224, 224), color="white")

inputs = processor(images=image, return_tensors="pt")

print("Running HF Inference...")
with torch.no_grad():
    # Gọi hàm lấy feature
    outputs = model.get_image_features(**inputs)

    # --- [FIX LỖI TẠI ĐÂY] ---
    # Kiểm tra xem output là Tensor hay Object để xử lý tương ứng
    if isinstance(outputs, torch.Tensor):
        hf_img = outputs
    elif hasattr(outputs, "image_embeds"):
        hf_img = outputs.image_embeds
    elif hasattr(outputs, "pooler_output"):
        hf_img = outputs.pooler_output
    else:
        # Trường hợp hiếm gặp khác, cố gắng lấy phần tử đầu tiên
        hf_img = outputs[0]

    # Bây giờ hf_img chắc chắn là Tensor, ta mới gọi .norm()
    hf_img = hf_img / hf_img.norm(dim=-1, keepdim=True)

# Chuyển sang Numpy để so sánh
hf_img_numpy = hf_img.cpu().numpy()

print("HF image embedding norm:", np.linalg.norm(hf_img_numpy))

# -----------------------------
# 3️⃣ Load ONNX
# -----------------------------
print("Loading ONNX...")
session = ort.InferenceSession(ONNX_PATH)

onnx_inputs = {
    "pixel_values": inputs["pixel_values"].numpy()
}

# Chạy ONNX
onnx_img = session.run(None, onnx_inputs)[0]

print("ONNX image embedding norm:", np.linalg.norm(onnx_img))

# -----------------------------
# 4️⃣ Cosine similarity HF vs ONNX
# -----------------------------
# Tính độ tương đồng giữa PyTorch gốc và ONNX export
# Kết quả càng gần 1.0 (hoặc 0.9999...) càng tốt
cos = np.dot(hf_img_numpy, onnx_img.T)

print("\nCosine similarity HF vs ONNX:", cos[0][0])

Loading HF model...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running HF Inference...
HF image embedding norm: 0.9999999
Loading ONNX...
ONNX image embedding norm: 1.0000001

Cosine similarity HF vs ONNX: 1.0


In [ ]:
texts = ["sunny street", "dog", "icy"]

# 2. Tokenize
text_inputs = processor(text=texts, return_tensors="pt", padding=True)

# 3. Get Features & Normalize
with torch.no_grad():
    # Gọi hàm lấy feature
    outputs = model.get_text_features(**text_inputs)

    # --- [FIX LỖI QUAN TRỌNG] ---
    # Kiểm tra xem output là Tensor hay Object để xử lý
    if isinstance(outputs, torch.Tensor):
        hf_text = outputs
    elif hasattr(outputs, "text_embeds"):
        hf_text = outputs.text_embeds
    elif hasattr(outputs, "pooler_output"):
        # Trường hợp này thường trả về 768 chiều (chưa project)
        # Nếu model là CLIPModel chuẩn thì hiếm khi vào đây
        hf_text = outputs.pooler_output
    else:
        # Fallback: Lấy phần tử đầu tiên nếu là tuple
        hf_text = outputs[0]

    # Bây giờ hf_text chắc chắn là Tensor, ta mới gọi .norm()
    hf_text = hf_text / hf_text.norm(dim=-1, keepdim=True)

# 4. Convert sang Numpy & In kết quả
hf_text = hf_text.numpy()

print(f"Shape: {hf_text.shape}") # Nên là (3, 512)
print("Text norms:", np.linalg.norm(hf_text, axis=1))
# Kết quả norm nên xấp xỉ 1.0 (ví dụ: [0.99999994

Shape: (3, 512)
Text norms: [0.99999994 1.         0.99999994]


In [ ]:
similarity = hf_img @ hf_text.T
similarity = similarity * model.logit_scale.exp().item()

print("\nSimilarity:", similarity)



Similarity: tensor([[20.1371, 21.7713, 20.2424]])


/tmp/ipython-input-3390674422.py:1: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  similarity = hf_img @ hf_text.T


In [ ]:
import onnx

model = onnx.load("assets/models/clip_vision_512.onnx")
print("Number of nodes:", len(model.graph.node))
print("Number of initializers:", len(model.graph.initializer))

Number of nodes: 499
Number of initializers: 200


In [ ]:
import os
print("Size:", os.path.getsize(ONNX_PATH)/1024/1024, "MB")


Size: 335.2328186035156 MB


In [ ]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession("assets/models/clip_vision_512.onnx")
dummy = np.random.randn(1,3,224,224).astype(np.float32)
out = session.run(None, {"pixel_values": dummy})[0]

print(out.shape)
print("Norm:", np.linalg.norm(out))

(1, 512)
Norm: 1.0000004


In [ ]:
import numpy as np
import os
import torch
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "openai/clip-vit-base-patch32"

keywords = [
    # Weather
    "rainy", "stormy", "sunny", "foggy", "snowy", "cloudy",
    "windy", "thunder", "lightning", "drizzle", "misty",
    "humid", "icy", "heatwave", "breeze",

    # Animals
    "wolf", "lion", "tiger", "cat", "dog", "eagle",
    "owl", "fox", "whale", "shark", "butterfly",
    "dragon", "snake", "horse", "crow",

    # Feelings
    "happy", "sad", "melancholic", "nostalgic", "romantic",
    "lonely", "hopeful", "angry", "peaceful", "anxious",
    "dreamy", "mysterious", "dark", "uplifting",
    "energetic", "calm", "chill", "epic", "intense",

    # Genres
    "lofi", "jazz", "classical", "rock", "indie",
    "ambient", "electronic", "pop", "hip hop",
    "trap", "orchestral", "acoustic", "techno",
    "house", "punk", "metal", "synthwave",

    # Scenes
    "late night city", "sunset beach", "forest morning",
    "empty road", "rainy window", "coffee shop",
    "mountain sunrise", "neon city", "space travel",
    "cyberpunk alley", "desert horizon", "underwater world",

    # Times
    "midnight", "dawn", "twilight", "evening",
    "morning", "afternoon"
]

print("⏳ Loading CLIPModel...")
model = CLIPModel.from_pretrained(MODEL_NAME)
model.eval()

processor = CLIPProcessor.from_pretrained(MODEL_NAME)

print("⏳ Tokenizing...")
inputs = processor(text=keywords, return_tensors="pt", padding=True)

print("⏳ Generating features...")
with torch.no_grad():
    # Gọi hàm lấy feature
    outputs = model.get_text_features(**inputs)

    # --- [FIX LỖI TẠI ĐÂY] ---
    # Kiểm tra xem nó là Tensor hay Object để xử lý
    if isinstance(outputs, torch.Tensor):
        text_features = outputs
    elif hasattr(outputs, "text_embeds"):
        text_features = outputs.text_embeds
    elif hasattr(outputs, "pooler_output"):
        text_features = outputs.pooler_output
    else:
        # Trường hợp đường cùng: lấy phần tử đầu tiên
        text_features = outputs[0]

    # Bây giờ text_features chắc chắn là Tensor, ta mới gọi .norm()
    # Normalize vector về độ dài 1 (để tính Cosine Similarity)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # Chuyển sang Numpy
    embeddings = text_features.detach().cpu().numpy().astype(np.float32)

# Lưu file
os.makedirs("scripts/artifacts", exist_ok=True)
np.save("scripts/artifacts/vibe_embeddings.npy", embeddings)
np.save("scripts/artifacts/vibe_keywords.npy", np.array(keywords))

print("-" * 30)
print(f"✅ DONE! Saved embeddings shape: {embeddings.shape}")
# Kiểm tra kích thước output (Nên là 512)
if embeddings.shape[1] == 512:
    print("=> Kích thước chuẩn 512. Khớp hoàn toàn với model ONNX ảnh.")
else:
    print(f"❌ CẢNH BÁO: Kích thước lạ ({embeddings.shape[1]}). Model ảnh và chữ sẽ không hiểu nhau!")
print("-" * 30)

⏳ Loading CLIPModel...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Tokenizing...
⏳ Generating features...
------------------------------
✅ DONE! Saved embeddings shape: (84, 512)
=> Kích thước chuẩn 512. Khớp hoàn toàn với model ONNX ảnh.
------------------------------


# Cũ

In [ ]:
import torch

from PIL import Image

from transformers import CLIPProcessor, CLIPModel



device = "cuda" if torch.cuda.is_available() else "cpu"



model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [ ]:
image = Image.open("/content/your_image.png")

inputs = processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
  image_outputs = model.get_image_features(**inputs)

if hasattr(image_outputs, "pooler_output"):
  image_features = image_outputs.pooler_output
else:
  image_features = image_outputs
# Normalize (QUAN TRỌNG)
image_embedding = image_features / image_features.norm(dim=-1, keepdim=True)

print(image_embedding.shape)

torch.Size([1, 512])


In [ ]:
texts = [
    # Weather
    "rainy", "stormy", "sunny", "foggy", "snowy", "cloudy",
    "windy", "thunder", "lightning", "drizzle", "misty",
    "humid", "icy", "heatwave", "breeze",

    # Animals
    "wolf", "lion", "tiger", "cat", "dog", "eagle",
    "owl", "fox", "whale", "shark", "butterfly",
    "dragon", "snake", "horse", "crow",

    # Feelings
    "happy", "sad", "melancholic", "nostalgic", "romantic",
    "lonely", "hopeful", "angry", "peaceful", "anxious",
    "dreamy", "mysterious", "dark", "uplifting",
    "energetic", "calm", "chill", "epic", "intense",

    # Genres
    "lofi", "jazz", "classical", "rock", "indie",
    "ambient", "electronic", "pop", "hip hop",
    "trap", "orchestral", "acoustic", "techno",
    "house", "punk", "metal", "synthwave",

    # Scenes
    "late night city", "sunset beach", "forest morning",
    "empty road", "rainy window", "coffee shop",
    "mountain sunrise", "neon city", "space travel",
    "cyberpunk alley", "desert horizon", "underwater world",

    # Times
    "midnight", "dawn", "twilight", "evening",
    "morning", "afternoon"
]
text_inputs = processor(text=texts, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
  text_outputs = model.get_text_features(**text_inputs)

if hasattr(text_outputs, "pooler_output"):
  text_features = text_outputs.pooler_output
else:
  text_features = text_outputs

text_features = text_features / text_features.norm(dim=-1, keepdim=True)

similarity = (image_embedding @ text_features.T)

print(similarity)

tensor([[0.2113, 0.1795, 0.2064, 0.1995, 0.1959, 0.2049, 0.2016, 0.1958, 0.1868,
         0.2116, 0.2099, 0.2214, 0.1862, 0.2187, 0.2087, 0.1827, 0.1877, 0.1865,
         0.1900, 0.1849, 0.1843, 0.1892, 0.1839, 0.1741, 0.1820, 0.2014, 0.1988,
         0.1913, 0.1902, 0.1942, 0.2008, 0.2085, 0.2135, 0.2364, 0.2218, 0.2248,
         0.2162, 0.1883, 0.2275, 0.1916, 0.2289, 0.2110, 0.1785, 0.2063, 0.1934,
         0.2143, 0.2043, 0.2133, 0.1969, 0.2435, 0.1992, 0.2008, 0.1926, 0.2130,
         0.2238, 0.1940, 0.2081, 0.1831, 0.2096, 0.2137, 0.2060, 0.1914, 0.2227,
         0.1941, 0.1855, 0.2089, 0.1907, 0.2223, 0.2108, 0.2418, 0.1787, 0.2212,
         0.1928, 0.2285, 0.1969, 0.2252, 0.2036, 0.1888, 0.2003, 0.2240, 0.2227,
         0.2154, 0.2094, 0.2147]])


In [ ]:
# 1. Lấy dòng đầu tiên (vì bạn chỉ có 1 ảnh) và chuyển về CPU/List
scores = similarity[0].cpu().tolist()

# 2. Ghép cặp (Text, Score) lại với nhau
results = list(zip(texts, scores))

# 3. Sắp xếp giảm dần (Score cao nhất lên đầu)
results.sort(key=lambda x: x[1], reverse=True)

# 4. In ra kết quả đẹp
print(f"{'Text':<15} | {'Score':<10}")
print("-" * 30)
for text, score in results:
    print(f"{text:<15} | {score:.4f}")

Text            | Score     
------------------------------
lofi            | 0.2435
empty road      | 0.2418
nostalgic       | 0.2364
dreamy          | 0.2289
neon city       | 0.2285
peaceful        | 0.2275
cyberpunk alley | 0.2252
lonely          | 0.2248
dawn            | 0.2240
ambient         | 0.2238
house           | 0.2227
twilight        | 0.2227
sunset beach    | 0.2223
romantic        | 0.2218
humid           | 0.2214
coffee shop     | 0.2212
heatwave        | 0.2187
hopeful         | 0.2162
evening         | 0.2154
afternoon       | 0.2147
calm            | 0.2143
orchestral      | 0.2137
melancholic     | 0.2135
epic            | 0.2133
indie           | 0.2130
drizzle         | 0.2116
rainy           | 0.2113
mysterious      | 0.2110
forest morning  | 0.2108
misty           | 0.2099
trap            | 0.2096
morning         | 0.2094
synthwave       | 0.2089
breeze          | 0.2087
sad             | 0.2085
pop             | 0.2081
sunny           | 0.2064
uplifting      